In [2]:
import duckdb
import pandas as pd

# Load dataset
users = pd.read_csv("../data/users.csv")
orders = pd.read_csv("../data/orders.csv")
orders["order_date"] = pd.to_datetime(orders["order_date"])
events = pd.read_csv("../data/events.csv")
events["event_ts"] = pd.to_datetime(events["event_ts"])

# Register dataframe as DuckDB table
con = duckdb.connect()
con.register("users", users)
con.register("orders", orders)
con.register("events", events)

In [3]:
completed_orders = orders[orders["order_status"] == "completed"]

orders_per_user = (
    completed_orders
    .groupby("user_id")["order_id"]
    .count()
    .reset_index()
)

orders_per_user["repeat_purchase"] = (orders_per_user["order_id"] >= 2).astype(int)

orders_per_user.head()

,user_id,order_id,repeat_purchase
0,1,1,0
1,2,2,1
2,8,1,0
3,11,1,0
4,13,1,0


In [4]:
user_features = (
    completed_orders
    .groupby("user_id")
    .agg(
        total_orders=("order_id","count"),
        total_revenue=("revenue","sum"),
        avg_order_value=("revenue","mean")
    )
    .reset_index()
)
user_features = user_features.merge(users, on="user_id")
user_features = user_features.merge(
    orders_per_user[["user_id","repeat_purchase"]],
    on="user_id"
)

user_features.head()

,user_id,total_orders,total_revenue,avg_order_value,signup_date,country,acquisition_channel,device,age_bucket,repeat_purchase
0,1,1,41.95,41.950,2025-05-13,SE,Affiliate,desktop,55+,0
1,2,2,84.17,42.085,2025-06-20,SE,Affiliate,mobile,45-54,1
2,8,1,43.43,43.430,2025-09-07,GB,Paid Search,mobile,25-34,0
3,11,1,51.96,51.960,2025-05-14,IT,Email,mobile,35-44,0
4,13,1,109.55,109.550,2025-05-30,GB,Paid Search,desktop,35-44,0


In [5]:
total_users = len(user_features)
repeat_users = user_features["repeat_purchase"].sum()

print("Total users:", total_users)
print("Repeat purchasers:", repeat_users)
print("Repeat rate:", repeat_users / total_users)

Total users: 2882
Repeat purchasers: 1317
Repeat rate: 0.45697432338653715


##### Insight
~46% repeat
~54% non-repeat

------------------------------------------------------------------------------------------------------------------------------

In [6]:
user_time_features = (
    completed_orders
    .groupby("user_id")
    .agg(
        first_order=("order_date","min"),
        last_order=("order_date","max")
    )
    .reset_index()
)

user_time_features["customer_lifetime_days"] = (
    user_time_features["last_order"] - user_time_features["first_order"]
).dt.days

user_features = user_features.merge(user_time_features, on="user_id")

In [7]:
avg_customer_lifetime = int(user_features["customer_lifetime_days"].mean())
print("Average Customer Lifetime in days:", avg_customer_lifetime)

Average Customer Lifetime in days: 26


##### Insight
Most repeat purchases happen within about one month
Customer engagement is short-cycle, typical for retail/e-commerce
Retention strategies should focus on the first 30 days

-----------------------------------------------------------------------------------------------------------------------------

In [18]:
user_features.head()

,user_id,total_orders,total_revenue,avg_order_value,signup_date,country,acquisition_channel,device,age_bucket,repeat_purchase,first_order,last_order,customer_lifetime_days
0,1,1,41.95,41.950,2025-05-13,SE,Affiliate,desktop,55+,0,2025-05-17,2025-05-17,0
1,2,2,84.17,42.085,2025-06-20,SE,Affiliate,mobile,45-54,1,2025-07-06,2025-07-31,25
2,8,1,43.43,43.430,2025-09-07,GB,Paid Search,mobile,25-34,0,2025-09-25,2025-09-25,0
3,11,1,51.96,51.960,2025-05-14,IT,Email,mobile,35-44,0,2025-05-17,2025-05-17,0
4,13,1,109.55,109.550,2025-05-30,GB,Paid Search,desktop,35-44,0,2025-07-08,2025-07-08,0


##### Query
We will build a Logistic Regression model to predict repeat purchase.

In [31]:
first_orders = (
    completed_orders
    .sort_values("order_date")
    .groupby("user_id")
    .first()
    .reset_index()
)

first_orders.head()

,user_id,order_id,order_date,basket_subtotal,discount,shipping_fee,tax,revenue,payment_method,order_status
0,1,1,2025-05-17,43.40,15.19,8.53,5.21,41.95,apple_pay,completed
1,2,3,2025-07-06,37.86,0.00,5.77,4.54,48.17,paypal,completed
2,8,6,2025-09-25,46.95,16.43,7.28,5.63,43.43,card,completed
3,11,7,2025-05-17,42.98,0.00,3.82,5.16,51.96,card,completed
4,13,8,2025-07-08,95.52,0.00,2.57,11.46,109.55,klarna,completed


In [32]:
first_orders = first_orders.merge(
    orders_per_user[["user_id","repeat_purchase"]],
    on="user_id"
)
first_orders.head()

,user_id,order_id,order_date,basket_subtotal,discount,shipping_fee,tax,revenue,payment_method,order_status,repeat_purchase
0,1,1,2025-05-17,43.40,15.19,8.53,5.21,41.95,apple_pay,completed,0
1,2,3,2025-07-06,37.86,0.00,5.77,4.54,48.17,paypal,completed,1
2,8,6,2025-09-25,46.95,16.43,7.28,5.63,43.43,card,completed,0
3,11,7,2025-05-17,42.98,0.00,3.82,5.16,51.96,card,completed,0
4,13,8,2025-07-08,95.52,0.00,2.57,11.46,109.55,klarna,completed,0


In [33]:
first_orders = first_orders.merge(
    users[["user_id","acquisition_channel","device","country"]],
    on="user_id"
)
first_orders.head()

,user_id,order_id,order_date,basket_subtotal,discount,shipping_fee,tax,revenue,payment_method,order_status,repeat_purchase,acquisition_channel,device,country
0,1,1,2025-05-17,43.40,15.19,8.53,5.21,41.95,apple_pay,completed,0,Affiliate,desktop,SE
1,2,3,2025-07-06,37.86,0.00,5.77,4.54,48.17,paypal,completed,1,Affiliate,mobile,SE
2,8,6,2025-09-25,46.95,16.43,7.28,5.63,43.43,card,completed,0,Paid Search,mobile,GB
3,11,7,2025-05-17,42.98,0.00,3.82,5.16,51.96,card,completed,0,Email,mobile,IT
4,13,8,2025-07-08,95.52,0.00,2.57,11.46,109.55,klarna,completed,0,Paid Search,desktop,GB


In [34]:
features = [
    "revenue",
]

X = first_orders[features]
y = first_orders["repeat_purchase"]

In [35]:
first_orders_encoded = pd.get_dummies(
    first_orders[["revenue","acquisition_channel","device","country"]],
    drop_first=True
)

X = first_orders_encoded
y = first_orders["repeat_purchase"]

In [36]:
event_features = (
    events
    .groupby("user_id")
    .agg(
        num_views=("event_type", lambda x: (x=="view").sum()),
        num_add_to_cart=("event_type", lambda x: (x=="add_to_cart").sum()),
        num_purchase_events=("event_type", lambda x: (x=="purchase").sum())
    )
    .reset_index()
)
first_orders = first_orders.merge(event_features, on="user_id")
first_orders.head()

,user_id,order_id,order_date,basket_subtotal,discount,shipping_fee,tax,revenue,payment_method,order_status,repeat_purchase,acquisition_channel,device,country,num_views,num_add_to_cart,num_purchase_events
0,1,1,2025-05-17,43.40,15.19,8.53,5.21,41.95,apple_pay,completed,0,Affiliate,desktop,SE,14,3,1
1,2,3,2025-07-06,37.86,0.00,5.77,4.54,48.17,paypal,completed,1,Affiliate,mobile,SE,2,0,2
2,8,6,2025-09-25,46.95,16.43,7.28,5.63,43.43,card,completed,0,Paid Search,mobile,GB,10,1,1
3,11,7,2025-05-17,42.98,0.00,3.82,5.16,51.96,card,completed,0,Email,mobile,IT,5,1,1
4,13,8,2025-07-08,95.52,0.00,2.57,11.46,109.55,klarna,completed,0,Paid Search,desktop,GB,0,0,1


In [37]:
first_orders_encoded = pd.get_dummies(
    first_orders[["revenue","acquisition_channel","device","country","num_views","num_add_to_cart"]],
    drop_first=True
)

X = first_orders_encoded
y = first_orders["repeat_purchase"]

In [39]:
first_orders.head()

,user_id,order_id,order_date,basket_subtotal,discount,shipping_fee,tax,revenue,payment_method,order_status,repeat_purchase,acquisition_channel,device,country,num_views,num_add_to_cart,num_purchase_events
0,1,1,2025-05-17,43.40,15.19,8.53,5.21,41.95,apple_pay,completed,0,Affiliate,desktop,SE,14,3,1
1,2,3,2025-07-06,37.86,0.00,5.77,4.54,48.17,paypal,completed,1,Affiliate,mobile,SE,2,0,2
2,8,6,2025-09-25,46.95,16.43,7.28,5.63,43.43,card,completed,0,Paid Search,mobile,GB,10,1,1
3,11,7,2025-05-17,42.98,0.00,3.82,5.16,51.96,card,completed,0,Email,mobile,IT,5,1,1
4,13,8,2025-07-08,95.52,0.00,2.57,11.46,109.55,klarna,completed,0,Paid Search,desktop,GB,0,0,1


In [40]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [41]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression()

model.fit(X_train, y_train)

C:\Users\bthn2\anaconda3\lib\site-packages\sklearn\linear_model\_logistic.py:814: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression()

In [44]:
from sklearn.metrics import accuracy_score

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Model accuracy:", accuracy)

Model accuracy: 0.587521663778163


In [43]:
baseline = max(
    y_test.mean(),
    1 - y_test.mean()
)

print("Baseline accuracy:", baseline)

Baseline accuracy: 0.5528596187175043


In [45]:
importance = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
})

importance.sort_values("coefficient", ascending=False)

,feature,coefficient
14,country_IE,0.378997
7,acquisition_channel_Paid Search,0.294518
6,acquisition_channel_Organic Search,0.263304
18,country_SE,0.247178
12,country_FR,0.178999
2,num_add_to_cart,0.164019
5,acquisition_channel_Email,0.143823
15,country_IT,0.117218
3,acquisition_channel_Direct,0.107815
1,num_views,0.016083


##### Insight


--------------------------------------------------------------------------------------------------------------------------------